In [ ]:
import numpy as np 
import math
import os
from tqdm.notebook import tqdm
from scipy.stats import ranksums, kruskal
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib as mlt
import matplotlib.lines as mlines
import glob
import matplotlib.pyplot as plt
import pandas as pd

from numpy import mean
from numpy import var

from matplotlib import cm
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms


In [ ]:

def compute_bootstraped_percentile(data, nTrials):
    nSubj, nFreqs = data.shape
    bootMat = np.zeros((nFreqs, nTrials))
    for i_trials in range(nTrials):
        idx_subj = np.random.randint(0,nSubj,nSubj)
        bootMat[:,i_trials] = np.nanmean(data[idx_subj,:], axis=0)
    return np.percentile(bootMat, (2.5,97.5), axis=1)

def plot_curve(ax, freq_vals, data, color, ls, label, a_size=8, alpha=1):
            
    def compute_bootstraped_percentile(data, nTrials):
        nSubj, nFreqs = data.shape
        bootMat = np.zeros((nFreqs, nTrials))
        for i_trials in range(nTrials):
            idx_subj = np.random.randint(0,nSubj,nSubj)
            bootMat[:,i_trials] = np.nanmean(data[idx_subj,:], axis=0)
        return np.percentile(bootMat, (2.5,97.5), axis=1)
    
    perc = compute_bootstraped_percentile(data,1000)
    
    ax.fill_between(freq_vals, perc[0,...], perc[1,...], alpha=0.1, color=color)
    
    ax.semilogx(freq_vals, data.mean(axis=0), color=color, label=label, linestyle=ls)

    ax.tick_params(labelsize=8)

        
def cohend(d1, d2, type='rm'):
    # calculate the size of samples
    n1, n2 = len(d1), len(d2)
    # calculate the variance of the samples
    s1, s2 = var(d1, ddof=1), var(d2, ddof=1)
    # calculate the pooled standard deviation
    if type=='rm':
        difference = d1 - d2
        s = np.std(difference)
    else:
        s = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
            
    # calculate the means of the samples
    u1, u2 = mean(d1), mean(d2)
    # calculate the effect size
    return (u1 - u2) / s

def plot_cohen(ax, freq, stats, pos, color='black'):

    for idxd, d in enumerate(stats):
        
        if d > 0.8: # large effect
            
            ax.scatter(freq[idxd], pos, marker='s', edgecolors=color, facecolors='none')
            
        if d > 0.5: # medium effect
            
            ax.scatter(freq[idxd], pos, marker='s', edgecolors=color, facecolors='none')

In [ ]:
def create_frequency_axis(f_min=2, f_max= 4, num_wavelet = 10):

    f_min_log = math.floor(np.log10(f_min))
    f_max_log = math.ceil(np.log10(f_max))
    mb = np.logspace(f_min_log, f_max_log, num=num_wavelet) # Morlet bank 
    freq_vals = mb[(2.1 < mb) & (mb < 90)] # Frequency of interest
    
    freq_labels = [('%.2f'%x) for x in freq_vals]
    
    return freq_vals, freq_labels
    
def extract_fei_new(files):
    
    name_subjs, data_masked = list(), list()
    for _, file in enumerate(tqdm(files)):
        
        name_subjs.append(file.split('/')[9][:6])
        
        tmp = np.load(file, allow_pickle=True)

        data_masked.append(np.nanmean(tmp[1,...],axis=0)) # tmp[1] == rimuovo fEI dove DFA < 0.6
    
    fEI_masked = np.array(data_masked)

    return name_subjs, fEI_masked

def extract_bis_new(files):
    
    data, name_subjs = list(), list()
    for ifile, file in enumerate(tqdm(files)):
        
        name_subjs.append(file.split('/')[9][:6])
        
        tmp = np.load(file, allow_pickle=True)
        
        data.append(np.mean(tmp,axis=0)) # Average channels
    
    bis = np.array(data)
    return name_subjs, bis

def extract_dfa_new(files):
    
    dfa, name_subjs = list(), list()
    for ifile, file in enumerate(tqdm(files)):
      
        name_subjs.append(file.split('/')[9][:6])
        datafile = np.load(file, allow_pickle=True)
        data = datafile.item()
        data_dfa = data['DFA']
        dfa.append(np.nanmean(data_dfa,axis=0))
    
    dfa = np.array(dfa)
    return name_subjs, dfa

In [ ]:
# Set frequency limits
f_min = 2
f_max = 90
num_wavelet=40

freq_vals, _ = create_frequency_axis(f_max=f_max, f_min=f_min, num_wavelet=num_wavelet)
freq_vals = freq_vals[:30]

In [ ]:
name_subj_r1, dfa_r1 = extract_dfa_new(sorted(glob.glob(os.path.join('dfa_r1'))))
name_subj_r2, dfa_r2 = extract_dfa_new(sorted(glob.glob(os.path.join('dfa_r2'))))
_, dfa_C = extract_dfa_new(sorted(glob.glob(os.path.join('dfa_ctrl'))))

_, fei_r1 = extract_fei_new(sorted(glob.glob(os.path.join('fei_r1'))))
_, fei_r2 = extract_fei_new(sorted(glob.glob(os.path.join('fei_r2'))))
_, fei_C = extract_fei_new(sorted(glob.glob(os.path.join('fei_ctrl'))))

_, bis_r1 = extract_bis_new(sorted(glob.glob(os.path.join('bis_r1'))))
_, bis_r2 = extract_bis_new(sorted(glob.glob(os.path.join('bis_r2'))))
_, bis_C = extract_bis_new(sorted(glob.glob(os.path.join('bis_ctrl'))))


In [ ]:
# Import LOOK UP TABLE - RBD 

lut_rbd = pd.read_excel('fname_rbd').set_index('ID')

lut_baseline = lut_rbd[lut_rbd['REC'] == 'B']
lut_fup = lut_rbd[lut_rbd['REC'] == 'FU1']

# Import LOOK UP TABLE - RBD 
lut_ctrl = pd.read_excel('fname_ctrl').set_index('ID')

In [ ]:
agle_lst = list()
name_lst = list()
for name in lut_fup.index:
    name_base = name.replace('run-02', 'run-01')
    name_lst.append(name_base)

In [ ]:
lut_base_r1r2 = lut_baseline.loc[name_lst]
idx_r1r2 = [name_subj_r1.index(x) for x in name_subj_r2 if x in name_subj_r1]

In [ ]:
def extract_data(data1, data2):
    delta = np.concatenate((data1[:,:7].mean(axis=1), data2[:,:7].mean(axis=1)))
    theta = np.concatenate((data1[:,7:11].mean(axis=1), data2[:,7:11].mean(axis=1)))
    alpha = np.concatenate((data1[:,11:16].mean(axis=1), data2[:,11:16].mean(axis=1)))
    beta = np.concatenate((data1[:,16:23].mean(axis=1), data2[:,16:23].mean(axis=1)))
    gamma = np.concatenate((data1[:,24:].mean(axis=1), data2[:,24:].mean(axis=1)))
    
    return np.concatenate((delta, theta, alpha, beta, gamma), axis=0)

In [ ]:
fei = extract_data(fei_r1, fei_C)
fei_fup = extract_data(fei_r1[idx_r1r2,:], fei_r2)

bis = extract_data(bis_r1, bis_C)
bis_fup = extract_data(bis_r1[idx_r1r2,:], bis_r2)

dfa = extract_data(dfa_r1, dfa_C)
dfa_fup = extract_data(dfa_r1[idx_r1r2,:], dfa_r2)



In [ ]:
nR = [len(name_subj_r1), fei_C.shape[0]]
nR_r1r2 = [len(name_subj_r2), len(name_subj_r2)]

groups_lst, groups_lst_r1r2, freqs_lst, freqs_lst_r1r2 = list(), list(), list(), list()

freqs = ['delta', 'theta', 'alpha', 'beta', 'gamma']
for iF in range(len(freqs)):
    freqs_lst.extend([freqs[iF]] * (nR[0] + nR[1]))

freqs = ['delta', 'theta', 'alpha', 'beta', 'gamma']
for iF in range(len(freqs)):
    freqs_lst_r1r2.extend([freqs[iF]] * (nR_r1r2[0] + nR_r1r2[1]))

groups = ['RBD', 'HC']
for iG in range(len(groups)):
    groups_lst.extend([groups[iG]] * nR[iG])

groups_r1r2 = ['B', 'FUP']
for iG in range(len(groups_r1r2)):
    groups_lst_r1r2.extend([groups_r1r2[iG]] * nR_r1r2[iG])

age_lst = lut_baseline['Age'].tolist() + lut_ctrl['Age'].tolist()
age_lst_r1r2 = lut_base_r1r2['Age'].tolist() + lut_fup['Age'].tolist()



gender_lst = lut_baseline['Sex'].tolist() + lut_ctrl['Sex'].tolist()

gender_lst_r1r2 = lut_base_r1r2['Sex'].tolist() + lut_fup['Sex'].tolist()


In [ ]:
df = pd.DataFrame()

df['fEI'] = fei 
df['dfa'] = dfa
df['BiS'] = bis
df['Frequencies'] = freqs_lst
df['Groups'] = groups_lst*5
df['Age'] = age_lst*5
df['Sex'] = gender_lst*5

In [ ]:
df_fup = pd.DataFrame()

df_fup['fEI'] = fei_fup
df_fup['dfa'] = dfa_fup
df_fup['BiS'] = bis_fup
df_fup['Frequencies'] = freqs_lst_r1r2
df_fup['Groups'] = groups_lst_r1r2*5
df_fup['Age'] = age_lst_r1r2*5
df_fup['Sex'] = gender_lst_r1r2*5

In [ ]:
from statsmodels.genmod.families.links import Log
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import RobustScaler
from statsmodels.formula.api import glm

eeg_measures = ['dfa', 'BiS', 'fEI']


pvals = np.zeros((len(eeg_measures), len(freqs)))

for i, iF in enumerate(freqs): 
    ddf = df[df['Frequencies']==iF]
    scaler = RobustScaler()
    df_standardized = pd.DataFrame(scaler.fit_transform(ddf.iloc[:,:3]), columns=ddf.columns[:3])

    df_standardized['Groups'] = ddf['Groups'].tolist()
    df_standardized['Age'] = ddf['Age'].tolist()
    df_standardized['Sex'] = ddf['Sex'].tolist()

    for j, ieeg in enumerate(eeg_measures):
        formula = ieeg + " ~ Groups + Age + Sex"
        model = smf.glm(formula=formula, data=df_standardized, family=sm.families.Gaussian()).fit()

        print(model.summary2().tables[1])
        

In [ ]:

pvals = np.zeros((len(eeg_measures), len(freqs)))

for i, iF in enumerate(freqs): 
    ddf = df_fup[df_fup['Frequencies']==iF]
    scaler = RobustScaler()
    df_standardized = pd.DataFrame(scaler.fit_transform(ddf.iloc[:,:3]), columns=ddf.columns[:3])

    df_standardized['Groups'] = ddf['Groups'].tolist()
    df_standardized['Age'] = ddf['Age'].tolist()
    df_standardized['Sex'] = ddf['Sex'].tolist()

    for j, ieeg in enumerate(eeg_measures):
        formula = ieeg + " ~ Groups + Age + Sex"
        model = smf.glm(formula=formula, data=df_standardized, family=sm.families.Gaussian()).fit()

        print(model.summary2().tables[1])

#### Compute confidence interval (bootstrap)

In [ ]:
perc_dfa_r1 = compute_bootstraped_percentile(dfa_r1[:,:30],1000)
perc_dfa_r1r2 = compute_bootstraped_percentile(dfa_r1[idx_r1r2,:30],1000)
perc_dfa_r2 = compute_bootstraped_percentile(dfa_r2[:,:30],1000)
perc_dfa_C = compute_bootstraped_percentile(dfa_C[:,:30],1000)

perc_bis_r1 = compute_bootstraped_percentile(bis_r1[:,:30],1000)
perc_bis_r1r2 = compute_bootstraped_percentile(bis_r1[idx_r1r2,:30],1000)
perc_bis_r2 = compute_bootstraped_percentile(bis_r2[:,:30],1000)
perc_bis_C = compute_bootstraped_percentile(bis_C[:,:30],1000)

perc_fei_r1 = compute_bootstraped_percentile(fei_r1[:,:30],1000)
perc_fei_r1r2 = compute_bootstraped_percentile(fei_r1[idx_r1r2,:30],1000)
perc_fei_r2 = compute_bootstraped_percentile(fei_r2[:,:30],1000)
perc_fei_C = compute_bootstraped_percentile(fei_C[:,:30],1000)

#### Figure 1

In [ ]:
import seaborn as sns
cmap = sns.color_palette("colorblind")

In [ ]:
#Panels 
fig = plt.figure(figsize=(17/2.54, 10/2.54), layout='constrained')


gs = fig.add_gridspec(nrows=5, ncols=3)

ax1 = fig.add_subplot(gs[:2, 0])
ax2 = fig.add_subplot(gs[:2, 1])
ax3 = fig.add_subplot(gs[:2, 2])

ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])
ax6 = fig.add_subplot(gs[2, 2])

ax7 = fig.add_subplot(gs[3:, 0])
ax8 = fig.add_subplot(gs[3:, 1])
ax9 = fig.add_subplot(gs[3:, 2])

a_size=8
l_size=10

# Axis 1
plot_curve(ax1, freq_vals, dfa_r1[:,:30], cmap[0], '-', 'iRBD - baseline')
plot_curve(ax1, freq_vals, dfa_C[:,:30], cmap[7], '-', 'HC')
ax1.set_ylim([.55, .9])
ax1.set_ylabel('DFA', fontsize=l_size)


# Axis 2
plot_curve(ax2, freq_vals, bis_r1[:,:30], cmap[0], '-', 'iRBD - baseline')
plot_curve(ax2, freq_vals, bis_C[:,:30], cmap[7], '-', 'HC')
ax2.set_ylim([2, 5])
ax2.set_ylabel('BiS', fontsize=l_size)

# Axis 3
plot_curve(ax3, freq_vals, fei_r1[:,:30], cmap[0], '-', 'iRBD - baseline')
plot_curve(ax3, freq_vals, fei_C[:,:30], cmap[7], '-', 'HC')
ax3.set_ylim([.7, 1.1])
ax3.set_ylabel('fEI', fontsize=l_size)

ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['top'].set_visible(False)

ax3.legend(loc='lower right', fontsize=a_size, frameon=False)

# Cohen's d
d_bis = list()
d_fei = list()
d_dfa = list()

for iF in range(len(freq_vals)):

    d_bis.append(cohend(bis_r1[:,iF], bis_C[:,iF], None))
    d_fei.append(cohend(fei_r1[:,iF], fei_C[:,iF], None))
    d_dfa.append(cohend(dfa_r1[:,iF], dfa_C[:,iF], None))
    
    
d_dfa = np.asarray(d_dfa)
d_bis = np.asarray(d_bis)
d_fei = np.asarray(d_fei)

# Axis 4
ax4.semilogx(freq_vals, d_dfa, color='black')
ax4.semilogx(freq_vals, d_dfa, color='black')
ax5.semilogx(freq_vals, d_bis, color='black')
ax5.semilogx(freq_vals, d_bis, color='black')
ax6.semilogx(freq_vals, d_fei, color='black')
ax6.semilogx(freq_vals, d_fei,color='black')

ax4.fill_between(freq_vals, 1.5, -1, where=(d_dfa > .5), alpha=0.2, color='black')
ax5.fill_between(freq_vals, 1.5, -1, where=(d_bis > .5), alpha=0.2, color='black')
ax6.fill_between(freq_vals, 1.5, -1, where=(d_fei > .5), alpha=0.2, color='black')

ax4.spines['right'].set_visible(False)
ax4.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)
ax5.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)
ax6.spines['top'].set_visible(False)

ax4.set_yticks([-1, 0, 1])
ax5.set_yticks([-1, 0, 1])
ax6.set_yticks([-1, 0, 1])

ax1.set_xticks([], [])
ax2.set_xticks([], [])
ax3.set_xticks([], [])
ax4.set_xticks([], [])
ax5.set_xticks([], [])
ax6.set_xticks([], [])

ax4.tick_params(labelsize=a_size)
ax5.tick_params(labelsize=a_size)
ax6.tick_params(labelsize=a_size)

ax4.set_ylabel("Cohen's d", fontsize=l_size)

# Axis 7
plot_curve(ax7, freq_vals, dfa_r1[idx_r1r2,:30], cmap[0], '-', 'iRBD - baseline')
plot_curve(ax7, freq_vals, dfa_r2[:,:30], cmap[3], '-', 'iRBD - follow-up')
ax7.set_ylim([.55, .9])
ax7.set_ylabel('DFA', fontsize=l_size)

# Axis 8
plot_curve(ax8, freq_vals, bis_r1[idx_r1r2,:30], cmap[0], '-', 'iRBD - baseline')
plot_curve(ax8, freq_vals, bis_r2[:,:30], cmap[3], '-', 'iRBD - follow-up')
ax8.set_ylim([2, 5])
ax8.set_ylabel('BiS', fontsize=l_size)

# Axis 9
plot_curve(ax9, freq_vals, fei_r1[idx_r1r2,:30], cmap[0], '-', 'iRBD - baseline')
plot_curve(ax9, freq_vals, fei_r2[:,:30], cmap[3], '-', 'iRBD - follow-up')
ax9.set_ylim([.7, 1.1])
ax9.set_ylabel('fEI', fontsize=l_size)

ax7.spines['right'].set_visible(False)
ax7.spines['top'].set_visible(False)
ax8.spines['right'].set_visible(False)
ax8.spines['top'].set_visible(False)
ax9.spines['right'].set_visible(False)
ax9.spines['top'].set_visible(False)

ax7.set_xticks([2,5,10,20,50,70], [2,5,10,20,50,70])
ax8.set_xticks([2,5,10,20,50,70], [2,5,10,20,50,70])
ax9.set_xticks([2,5,10,20,50,70], [2,5,10,20,50,70])
ax7.set_xlabel('Frequencies [Hz]', fontsize=l_size)
ax8.set_xlabel('Frequencies [Hz]', fontsize=l_size)
ax9.set_xlabel('Frequencies [Hz]', fontsize=l_size)

ax9.legend(loc='lower right', fontsize=a_size, frameon=False)

fig.align_ylabels([(ax1,ax2,ax3),(ax4,ax5,ax6),(ax7,ax8,ax9)])